# Condition Prediction Colab

Notebook pentru **generare de predicții**, nu evaluare.

Colab face doar partea care cere GPU:
- încarcă Qwen3-4B + LoRA adapter FT din Drive;
- rulează pe `TEST_COMPLETE_PAIR_IDS`;
- salvează predicțiile în layout-ul repo-ului;
- exportă un zip cu predicțiile.

Evaluarea se rulează local cu `src/SOTA_EVALUATION/eval.py`, după ce aduci zip-ul în repo.


In [43]:
# Dependencies for local-model prediction.
%pip -q install -U \
  "transformers>=4.57.0,<5" \
  "accelerate>=1.11.0" \
  "peft>=0.17.0" \
  "bitsandbytes>=0.45.0" \
  "jsonschema" \
  "sentencepiece" \
  "protobuf>=5.29.1,<6" \
  "anthropic"


In [ ]:
# Repo checkout / refresh.
import os
import subprocess
import sys
from pathlib import Path

# Build the URL from parts so notebook frontends cannot turn it into markdown.
REPO_URL = "".join(["https://", "github.com/", "SoftNestSol/", "Medical-Notes.git"])
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Medical-Notes")
UPDATE_REPO = True
# Set True in Colab when /content checkout is dirty and git pull fails.
# This deletes only the ephemeral Colab clone, not Drive backups.
FRESH_CLONE = False

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def run_checked(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("COMMAND FAILED:", " ".join(str(part) for part in cmd))
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
        raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout, result.stderr)
    if result.stderr.strip():
        print(result.stderr.strip())
    return result

if IN_COLAB:
    os.chdir("/content")
    if FRESH_CLONE and REPO_DIR.exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    if not REPO_DIR.exists():
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])
    elif UPDATE_REPO:
        try:
            subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
        except subprocess.CalledProcessError as exc:
            print("git pull --ff-only failed; continuing with existing /content checkout.")
            print("If you need the latest notebook code, delete /content/Medical-Notes and rerun checkout.")
            print("error:", exc)
    os.chdir(REPO_DIR)
else:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "src" / "data_split.py").exists():
            os.chdir(candidate)
            break

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
SOTA = SRC / "SOTA_EVALUATION"
ICL = SRC / "ICL"
SCRIPTS = ROOT / "scripts"
for path in [SRC, SOTA, ICL, SCRIPTS]:
    sys.path.insert(0, str(path))

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
print("ROOT", ROOT)

git pull --ff-only failed; continuing with existing /content checkout.
If you need the latest notebook code, delete /content/Medical-Notes and rerun checkout.
error: Command '['git', '-C', '/content/Medical-Notes', 'pull', '--ff-only']' returned non-zero exit status 1.
ROOT /content/Medical-Notes


In [ ]:
# Config.
from data_split import TEST_COMPLETE_PAIR_IDS

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
CONDITION_NAME = "cond5_qwen3_4b_synth_claude_ft"
PRED_DIR = ROOT / "data" / "chiropractor_ro" / "predictions" / CONDITION_NAME
CONV_DIR = ROOT / "data" / "chiropractor_ro" / "conversations"

# Empty RUN_NAME = auto-pick latest Drive run containing an adapter folder.
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/Medical-Notes/ft-runs")
RUN_NAME = ""
LOCAL_ADAPTER_DIR = Path("/content/ft_adapter")

TEST_IDS = sorted(TEST_COMPLETE_PAIR_IDS, key=lambda x: int(x.replace("audio", "")))
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 1024
OVERWRITE_PREDICTIONS = False

# Export locations. Drive backup is best-effort; direct download is fallback.
PREDICTION_BACKUP_ROOT = Path("/content/drive/MyDrive/Medical-Notes/predictions")
DOWNLOAD_PREDICTIONS = True

print("TEST_IDS", len(TEST_IDS), TEST_IDS)
print("PRED_DIR", PRED_DIR)


In [ ]:
# Locate/copy LoRA adapter from Drive.
import shutil

try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped/failed:", repr(exc))

def find_adapter_dir() -> Path:
    if RUN_NAME:
        candidate = DRIVE_RUN_ROOT / RUN_NAME / "adapter"
        if not candidate.exists():
            raise FileNotFoundError(candidate)
        return candidate
    candidates = sorted(
        [path for path in DRIVE_RUN_ROOT.glob("*/adapter") if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No adapter folders under {DRIVE_RUN_ROOT}")
    return candidates[0]

adapter_source = find_adapter_dir()
if LOCAL_ADAPTER_DIR.exists():
    shutil.rmtree(LOCAL_ADAPTER_DIR)
shutil.copytree(adapter_source, LOCAL_ADAPTER_DIR)
print("adapter source", adapter_source)
print("local adapter", LOCAL_ADAPTER_DIR)
print("files", sorted(path.name for path in LOCAL_ADAPTER_DIR.iterdir()))

In [ ]:
# Load base model + LoRA adapter for condition 5 inference.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    token=False,
)
model = PeftModel.from_pretrained(base_model, str(LOCAL_ADAPTER_DIR))
model.eval()
print("loaded model + adapter")

In [ ]:
# Prediction helpers.
import json
import re
from claude_zero_shot import SYSTEM_PROMPT
from build_real_icl_examples import read_conversation_as_transcript
from parser import ParseError, SchemaError, parse_note

CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}

def make_messages(transcript: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "--- TRANSCRIEREA CONSULTAȚIEI ---\n\n" + transcript.strip()},
    ]

def generate_raw(transcript: str) -> str:
    prompt = tokenizer.apply_chat_template(
        make_messages(transcript),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def write_note(path: Path, note: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(note, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [ ]:
# Generate FT condition predictions on TEST_COMPLETE_PAIR_IDS.
failures = []
PRED_DIR.mkdir(parents=True, exist_ok=True)

for idx, conv_id in enumerate(TEST_IDS, start=1):
    out_path = PRED_DIR / f"{conv_id}.json"
    raw_path = PRED_DIR / f"{conv_id}.raw.txt"
    if out_path.exists() and not OVERWRITE_PREDICTIONS:
        print(f"[{idx}/{len(TEST_IDS)}] {conv_id} skip")
        continue

    transcript = read_conversation_as_transcript(CONV_DIR / f"{conv_id}.json")
    raw = generate_raw(transcript)
    raw_path.write_text(raw, encoding="utf-8")
    try:
        note = parse_note(raw)
    except (ParseError, SchemaError) as exc:
        print(f"[{idx}/{len(TEST_IDS)}] {conv_id} FAIL {type(exc).__name__}: {exc}")
        failures.append({"id": conv_id, "error": f"{type(exc).__name__}: {exc}", "raw_path": str(raw_path)})
        continue
    write_note(out_path, note)
    print(f"[{idx}/{len(TEST_IDS)}] {conv_id} saved")

(PRED_DIR / "_failures.json").write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")
print("done", len(TEST_IDS) - len(failures), "ok", len(failures), "failed")
print("PRED_DIR", PRED_DIR)


In [ ]:
# Export predictions: zip + Drive backup + optional browser download.
import shutil

if not PRED_DIR.exists():
    raise FileNotFoundError(PRED_DIR)

manifest = {
    "condition_name": CONDITION_NAME,
    "model_id": MODEL_ID,
    "adapter_dir": str(LOCAL_ADAPTER_DIR),
    "prediction_dir": str(PRED_DIR),
    "test_ids": TEST_IDS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "prompt_source": "src/ICL/claude_zero_shot.py:SYSTEM_PROMPT",
}
(PRED_DIR / "_prediction_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = shutil.make_archive(str(PRED_DIR), "zip", PRED_DIR)
print("prediction zip", zip_path)

# Best-effort Drive backup. If Drive auth fails in extension, use direct download below.
try:
    from google.colab import drive, files  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    if Path("/content/drive/MyDrive").exists():
        PREDICTION_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
        drive_zip = PREDICTION_BACKUP_ROOT / f"{CONDITION_NAME}.zip"
        shutil.copy2(zip_path, drive_zip)
        print("Drive prediction zip", drive_zip)
    if DOWNLOAD_PREDICTIONS:
        files.download(zip_path)
except Exception as exc:
    print("Drive/download step failed:", repr(exc))
    print("Zip still exists in Colab at:", zip_path)


## Local Evaluation

După ce aduci zip-ul în repo local, extrage-l astfel încât fișierele să ajungă în:

`data/chiropractor_ro/predictions/cond5_qwen3_4b_synth_claude_ft/`

Apoi rulează local:

```bash
python src/SOTA_EVALUATION/eval.py \
  --pred-dir data/chiropractor_ro/predictions/cond5_qwen3_4b_synth_claude_ft \
  --split test-complete \
  --skip-bertscore
```
